# Import Library

In [1]:
import sys
import os
import joblib

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import AUC

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report,roc_auc_score,accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('..'))
from src.preprocess import URLEmbeddingTransformer

c:\Users\Alvin\miniconda3\envs\dl_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# tf.keras.backend.clear_session()
tf.compat.v1.reset_default_graph()

In [6]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # Should return True if GPU version was installed

2.5.1+cu121
True


In [8]:
print(f"TensorFlow Version: {tf.__version__}")
physical_devices = tf.config.list_physical_devices('GPU')

if len(physical_devices) > 0:
    print(f"GPU Terdeteksi: {physical_devices}")
    print("Detail:", tf.config.experimental.get_device_details(physical_devices[0]))
else:
    print("Failed")


TensorFlow Version: 2.19.1
Failed


# A. Exploring Dataset

In [11]:
df = pd.read_csv('../data/data.csv')

df = df.drop_duplicates(subset=['URL'])
df = df.dropna(subset=['URL', 'ClassLabel'])

In [12]:
df.head(5)

,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
0,https://www.womensweekly.com.sg,31,0,3,1,3.461320,6.0,2,1,2,0.00000,0,0,0,0,3,0.000000,1.0
1,http://116.53.34.145:34075/i,28,1,3,0,3.645593,7.0,2,1,9,2.00000,0,15,0,0,2,53.571429,0.0
2,http://58.23.215.31:8765/wzoptup.exe,36,1,4,0,4.086049,8.0,2,1,7,11.44075,0,13,0,1,3,36.111111,0.0
3,https://www.dudpro.co.il,24,0,3,1,3.772055,6.0,2,1,2,0.00000,0,0,0,0,2,0.000000,1.0
4,http://117.201.113.115:53518/i,30,1,3,0,3.819549,11.2,2,1,9,2.00000,0,17,0,0,3,0.371737,0.0


In [13]:
df.describe()

,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
count,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000,80755.000000
mean,35.038375,0.485357,3.009597,0.405585,3.976264,7.737573,1.512488,1.010662,5.001560,8.418308,0.043180,8.518903,0.281927,0.147223,6.298805,17.944022,0.375419
std,16.742432,0.499789,0.947135,0.491008,0.306617,2.695811,0.625887,0.203610,2.919111,14.474565,0.203263,8.552348,0.449941,0.354330,5.350614,21.058840,0.484234
min,11.000000,0.000000,1.000000,0.000000,2.521641,1.600000,-1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,27.000000,0.000000,2.000000,0.000000,3.770942,5.917567,1.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000
50%,31.000000,0.000000,3.000000,0.000000,3.937326,8.000000,2.000000,1.000000,3.000000,2.000000,0.000000,10.000000,0.000000,0.000000,3.000000,0.371737,0.000000
75%,35.000000,1.000000,4.000000,1.000000,4.100817,8.284594,2.000000,1.000000,8.000000,7.000000,0.000000,15.000000,1.000000,0.000000,10.000000,40.740741,1.000000
max,470.000000,1.000000,19.000000,1.000000,5.871503,57.600000,5.000000,14.000000,10.000000,317.000000,1.000000,164.000000,1.000000,1.000000,36.000000,65.957447,1.000000


## A.1 Data Figure

## A.2 Extracting Insight